# nb02 — предсказание породы пампа в момент пересечения +5%

**Гипотеза (из nb01 §3):** порода различима онлайн — будущий монстр добегает до
+5% втрое быстрее и с большим объёмом. Если так, фильтр «не фейдить вероятных
монстров» вернёт диагональный edge nb01 §2 в исполнимую форму.

**Контракт честности:**
- Признаки — только известное на баре пересечения +5%: t05 (скорость), retr05
  (откат по пути), surg05 (объём у пересечения), surge15/r15 (триггер), liq.
  НЕ используем k/clen (длина кластера известна только после его конца) и
  ничего из будущего пути.
- Метка «монстр» (runup ≥ 25% за 1440м) — hindsight, поэтому обучение строго
  walk-forward по месяцам: предсказывая месяц M, модель видит только пампы,
  чьё окно 1440м ЗАКОНЧИЛОСЬ до начала M.
- Порог политики выбирается на DEV (2024-07..2025-06) и замораживается;
  VALID/TEST видят его один раз.

In [1]:
import sys; sys.path.insert(0, '.')
from _lab import *
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

X = pd.read_parquet('_out/pump_levels.parquet')
X['entry'] = pd.to_datetime(X['entry'], utc=True)
X = X[X.t05 > 0].copy()                      # дошедшие до +5%
X['monster'] = (X.runup >= 0.25).astype(int)
X['logliq'] = np.log10(X.liq.clip(lower=1))
FEATS = ['t05','retr05','surg05','surge15','r15','logliq']
X = X.dropna(subset=FEATS + ['fade05_60']).sort_values('entry').reset_index(drop=True)
def window(t):
    if t < pd.Timestamp('2025-07-01', tz='UTC'): return 'TRAIN'
    if t < pd.Timestamp('2026-02-01', tz='UTC'): return 'VALID'
    return 'TEST'
X['win'] = X.entry.map(window)
print('events (crossed +5%):', len(X), '| monster rate:',
      round(X.monster.mean()*100,1), '%')
print(X.groupby('win').monster.mean().round(3).to_string())

events (crossed +5%): 33755 | monster rate: 27.1 %
win
TEST     0.303
TRAIN    0.209
VALID    0.312


## 1. Walk-forward по месяцам

Каждый месяц M: обучение на пампах с `entry < M − 1 день` (чтобы окно 1440м
метки закончилось), предсказание на M. Первые полгода — разгон без предсказаний.

In [2]:
months = pd.period_range('2024-07', '2026-07', freq='M')
X['pred'] = np.nan
for m in months:
    m0 = pd.Timestamp(m.start_time, tz='UTC'); m1 = pd.Timestamp((m+1).start_time, tz='UTC')
    tr = X[X.entry < m0 - pd.Timedelta(days=1)]
    te = X[(X.entry >= m0) & (X.entry < m1)]
    if len(tr) < 500 or len(te) == 0: continue
    clf = HistGradientBoostingClassifier(max_iter=200, random_state=0)
    clf.fit(tr[FEATS], tr.monster)
    X.loc[te.index, 'pred'] = clf.predict_proba(te[FEATS])[:,1]
P = X.dropna(subset=['pred']).copy()
print('predicted events:', len(P))
for w in ('TRAIN','VALID','TEST'):
    s = P[P.win == w]
    if len(s): print(f'{w}: AUC {roc_auc_score(s.monster, s.pred):.3f}  (n={len(s)}, base {s.monster.mean()*100:.0f}%)')

predicted events: 30064
TRAIN: AUC 0.629  (n=8830, base 22%)
VALID: AUC 0.663  (n=11500, base 31%)
TEST: AUC 0.670  (n=9734, base 30%)


## 2. Монотонность: децили предсказания vs реальность

Если модель настоящая — доля монстров и fade-доходность должны монотонно
меняться по децилям предсказанной вероятности, на данных ВНЕ обучения.

In [3]:
P['dec'] = pd.qcut(P.pred, 10, labels=False, duplicates='drop')
t = P.groupby('dec').agg(n=('monster','size'),
        monster_pct=('monster', lambda s: round(s.mean()*100,1)),
        fade60=('fade05_60', lambda s: round(s.mean()*100,2)),
        fade240=('fade05_240', lambda s: round(s.mean()*100,2)))
print('децили P(monster), все walk-forward предсказания:')
print(t.to_string())

децили P(monster), все walk-forward предсказания:
        n  monster_pct  fade60  fade240
dec                                    
0    3007         10.1   -0.25    -0.42
1    3006         13.6   -0.11    -0.50
2    3006         18.6   -0.30    -0.51
3    3007         22.4   -0.03    -0.08
4    3006         27.1   -0.23    -0.03
5    3006         30.9   -0.12     0.02
6    3007         33.1   -0.29    -0.50
7    3006         37.6   -0.70    -1.16
8    3006         40.6   -0.73    -0.92
9    3007         47.5   -1.25    -1.82


## 3. Политика: фейдим только «не-монстров». Порог заморожен на DEV

DEV = 2024-07..2025-06 (walk-forward предсказания внутри TRAIN-эпохи).
Свипуем порог только там, выбираем, применяем к VALID/TEST без изменений.
Сравнение — против «фейдить всех» (baseline nb01: ≈ 0).

In [4]:
DEV = P[P.entry < pd.Timestamp('2025-07-01', tz='UTC')]
print('=== свип порога на DEV ===')
print(f'{"порог":>6} {"n":>6} {"доля":>5} {"fade60 mean":>11} {"fade240 mean":>12}')
for thr in (0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 1.01):
    s = DEV[DEV.pred < thr]
    lab = 'все' if thr > 1 else f'{thr:.2f}'
    if len(s) > 50:
        print(f'{lab:>6} {len(s):>6} {len(s)/len(DEV)*100:>4.0f}% {s.fade05_60.mean()*100:>+10.2f}% {s.fade05_240.mean()*100:>+11.2f}%')

=== свип порога на DEV ===
 порог      n  доля fade60 mean fade240 mean
  0.10   3092   35%      -0.19%       -0.42%
  0.15   4064   46%      -0.21%       -0.42%
  0.20   5012   57%      -0.16%       -0.33%
  0.25   5913   67%      -0.13%       -0.25%
  0.30   6739   76%      -0.09%       -0.16%
  0.40   7792   88%      -0.06%       -0.14%
   все   8830  100%      -0.12%       -0.24%


In [5]:
THR = 0.20   # заморожено по DEV до чтения VALID/TEST
print(f'=== порог {THR}, применён к VALID и TEST один раз ===')
for w in ('VALID','TEST'):
    s = P[P.win == w]
    f = s[s.pred < THR]
    print(f'{w}: всех {len(s)} (fade60 {s.fade05_60.mean()*100:+.2f}%) | '
          f'фильтр {len(f)} = {len(f)/len(s)*100:.0f}% '
          f'(fade60 {f.fade05_60.mean()*100:+.2f}%, fade240 {f.fade05_240.mean()*100:+.2f}%, '
          f'монстров пропущено {f.monster.mean()*100:.1f}%)')

=== порог 0.2, применён к VALID и TEST один раз ===
VALID: всех 11500 (fade60 -0.89%) | фильтр 3313 = 29% (fade60 -0.34%, fade240 -0.53%, монстров пропущено 14.9%)
TEST: всех 9734 (fade60 -0.08%) | фильтр 2649 = 27% (fade60 -0.01%, fade240 -0.36%, монстров пропущено 14.1%)


In [6]:
# персист walk-forward предсказаний для nb03+ (порода как фильтр риска)
P[['sym','entry','pred','monster','win','runup']].to_parquet('_out/breed_preds.parquet')
print('saved', len(P), '-> _out/breed_preds.parquet')

saved 30064 -> _out/breed_preds.parquet


## Выводы nb02

**1. 🟢 Порода предсказуема — модель работает как сортировщик риска.**
Walk-forward AUC 0.63/0.66/0.67 по эпохам, децили строго монотонны:
в «безопасном» дециле 10% монстров, в опасном 47.5%; fade60 −0.25% vs −1.25%.
Гипотеза nb01 §3 подтверждена: скорость+объём на +5% несут сигнал о породе.

**2. 🔴 Но стратегией это НЕ становится: даже лучший дециль отрицателен.**
Свип порога на DEV: все варианты −0.06…−0.21%. Замороженный порог 0.20 на
VALID/TEST: −0.34%/−0.01% — лучше, чем фейдить всех (VALID −0.89%), но не >0.

**3. Почему: диагональ nb01 недостижима сортировкой по P(monster) в точке +5%.**
Диагональ («фейд G0 у вершины +3.3%») - это фейд ВОЗЛЕ ПИКА. А пересечение +5% —
это середина бега: даже правильно предсказанный не-монстр (медленный G1) имеет
впереди ещё +5-10% хода за следующие часы — фейд на 60м его не переживает.
Предсказать «кто» — мало; надо предсказать «где пик» или дождаться его.

**4. Куда это указывает — и это же говорят два независимых источника:**
вход не по уровню, а по ИСТОЩЕНИЮ. Старая линия pump закрылась ровно на этой
ноте («единственный честный путь — вход после первого lower-high»), и на
скриншоте практиков вход тоже у вершины после остановки роста, не в середине
бега. Порода при этом остаётся полезной — как фильтр риска (п.1).

**nb03: вход по истощению × предсказанная порода.** Определения истощения:
(а) первые N минут без нового максимума после пересечения +5%;
(б) подтверждённый конец кластера (10 тихих минут — уже есть в детекте).
Сравнить оба, с фильтром породы и без, DEV→VALID/TEST тем же протоколом.

**Оговорки.** Признаков всего 6, все с минутных свечей — у практиков
микроданные, потолок различимости у нас ниже. База монстров нестационарна
(22%→31%) — walk-forward это учёл, фиксированный порог вероятности - грубо.